In [69]:
from dotenv import load_dotenv

load_dotenv()

True

# Load MCP Server

In [70]:
import sys
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient

# Windows + Jupyter fix
if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

client = MultiServerMCPClient(
    {
        "local_server": {
            "transport": "stdio",
            "command": sys.executable,   # Don't use "python"
            "args": ["../resources/2.1_mcp_server.py"],
        }
    }
)

async def main():
    tools = await client.get_tools()
    print(tools)

await main()

[StructuredTool(name='search_web', description='Search the web for information', args_schema={'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'search_webArguments', 'type': 'object'}, handle_tool_error=<function _handle_mcp_tool_error at 0x1073a7600>, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x16d821440>)]


In [71]:
# get tools

tools = await client.get_tools()

#get resources
resources = await client.get_resources("local_server")

#get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [72]:
from langchain.agents import create_agent

agent = create_agent(
    model="ollama:qwen2.5:3b",
    tools=tools,
    system_prompt=prompt # type: ignore
) # type: ignore

In [73]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [74]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='4d94b160-31bf-46aa-95d4-4743e36b73b5'),
              AIMessage(content="I'm sorry, I can't find any information directly related to 'langchain-mcp-adapters' library. Let's try to find more details by searching the web.\n", additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-07-13T17:27:27.824945Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2808670333, 'load_duration': 169193500, 'prompt_eval_count': 274, 'prompt_eval_duration': 71292000, 'eval_count': 60, 'eval_duration': 2524585000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019f5c84-f447-73f2-b4f8-246cd961b147-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters'}, 'id': 'ab77f49e-3859-40ca-a2c2-d9e4ae60dc40', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens'

# Online MCP

In [75]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [76]:
agent = create_agent(
    model="ollama:qwen2.5:3b",
    tools=tools,
)

In [77]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='bedc4826-bcdb-49e0-bdb3-9f8a6da5dae8'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-07-13T17:27:50.436791Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2463702833, 'load_duration': 143433625, 'prompt_eval_count': 351, 'prompt_eval_duration': 1373193000, 'eval_count': 24, 'eval_duration': 910759000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019f5c85-4e01-7b90-8f60-6b18b220f73b-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'America/New_York'}, 'id': 'b24e90d1-a5ee-4c57-9a83-07f08189cef3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 351, 'output_tokens': 24, 'total_tokens': 375}),
              ToolMessage(content=[{'type': 'text', 'text': '{\n  "timezone": "America/New_York",\n  "datetime": "2026-07-13T

In [78]:
print(response["messages"][-1].content)

The current time in your local timezone (America/New_York) is:
- Date/Time: July 13, 2026 at 13:27 UTC-04:00
- Day of the Week: Monday
- Whether DST (Daylight Saving Time): Yes

Please note that this information might differ if you are in a different timezone. Would you like to know the time in another specific timezone?
